## CS310 Natural Language Processing
## Assignment 1. Neural Network-based Text Classification

**Total points**: 30

You should roughtly follow the structure of the notebook. Add additional cells if you feel needed. 

You can (and you should) re-use the code from Lab 2. 

Make sure your code is readable and well-structured.

### 0. Import Necessary Libraries

This notebook keeps the Lab 2 workflow, but adapts the data loading, tokenization, and evaluation logic to the Chinese humor detection dataset.

In [ ]:
import copy
import importlib.util
import json
import random
import re
from collections import Counter
from pathlib import Path

try:
    import jieba
except ImportError as exc:
    raise ImportError(
        "jieba is required for the improved tokenizer. Install dependencies from requirements_a1.txt before running this notebook."
    ) from exc

try:
    import torch
except ImportError as exc:
    raise ImportError(
        "PyTorch is required for this notebook. Install dependencies from requirements_a1.txt before running this notebook."
    ) from exc

from torch import nn
from torch.utils.data import DataLoader, Dataset

SEED = 310
random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if torch.backends.cudnn.is_available():
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def find_submission_dir(start=None):
    start = (start or Path.cwd()).resolve()
    search_roots = [start, *start.parents]

    required_files = ("A1_nn.ipynb", "train.jsonl", "test.jsonl", "data_utils.py")
    for candidate in search_roots:
        if all((candidate / name).exists() for name in required_files):
            return candidate

    for root in search_roots:
        for notebook_path in sorted(root.rglob("A1_nn.ipynb")):
            candidate = notebook_path.parent
            if all((candidate / name).exists() for name in required_files[1:]):
                return candidate

    raise FileNotFoundError(
        "Could not locate the submission directory containing A1_nn.ipynb, train.jsonl, test.jsonl, and data_utils.py."
    )


def load_local_module(module_name, module_path):
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not load module from {module_path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


A1_DIR = find_submission_dir()
TRAIN_PATH = A1_DIR / "train.jsonl"
TEST_PATH = A1_DIR / "test.jsonl"
DATA_UTILS_PATH = A1_DIR / "data_utils.py"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

a1_data_utils = load_local_module("a1_data_utils", DATA_UTILS_PATH)
build_vocab_from_iter = a1_data_utils.build_vocab_from_iter

print(f"Current working directory: {Path.cwd().resolve()}")
print(f"Submission directory: {A1_DIR}")
print(f"Device: {device}")
print(f"Random seed: {SEED}")

### 1. Data Processing

The JSONL files are loaded into a simple dataset format. I compare a basic Chinese-character tokenizer against a jieba-based tokenizer that preserves English words, digit sequences, and punctuation.

In [ ]:
CHINESE_CHAR_RE = re.compile(r"[一-鿿]")
TOKEN_PATTERN = re.compile(r"[A-Za-z]+|\d+|[一-鿿]+|[^\w\s]", re.UNICODE)


class HumorDataset(Dataset):
    def __init__(self, examples):
        self.examples = examples

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, index):
        return self.examples[index]


def normalize_label(label):
    if isinstance(label, list):
        if len(label) != 1:
            raise ValueError(f"Expected a single label, but got {label}")
        label = label[0]
    return int(label)


def load_jsonl(path):
    examples = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            examples.append(
                {
                    "id": row["id"],
                    "sentence": row["sentence"],
                    "label": normalize_label(row["label"]),
                }
            )
    return examples


def basic_tokenizer(text):
    return CHINESE_CHAR_RE.findall(text)


def improved_tokenizer(text):
    tokens = []
    for piece in TOKEN_PATTERN.findall(text):
        if piece.isspace():
            continue
        if re.fullmatch(r"[一-鿿]+", piece):
            tokens.extend(token for token in jieba.lcut(piece) if token.strip())
        elif re.fullmatch(r"[A-Za-z]+", piece):
            tokens.append(piece.lower())
        else:
            tokens.append(piece)
    return tokens


def yield_tokens(examples, tokenizer):
    for example in examples:
        yield tokenizer(example["sentence"])


def numericalize(text, tokenizer, vocab):
    token_ids = vocab(tokenizer(text))
    if not token_ids:
        return [vocab["<unk>"]]
    return token_ids


def stratified_split(examples, valid_ratio=0.1, seed=SEED):
    rng = random.Random(seed)
    by_label = {}
    for example in examples:
        by_label.setdefault(example["label"], []).append(example)

    train_split, valid_split = [], []
    for label, group in sorted(by_label.items()):
        group = group.copy()
        rng.shuffle(group)
        valid_size = max(1, int(len(group) * valid_ratio))
        valid_split.extend(group[:valid_size])
        train_split.extend(group[valid_size:])

    rng.shuffle(train_split)
    rng.shuffle(valid_split)
    return train_split, valid_split


def build_collate_fn(tokenizer, vocab, device):
    def collate_batch(batch):
        label_list = []
        token_ids_list = []
        offsets = [0]

        for example in batch:
            label_list.append(example["label"])
            token_ids = torch.tensor(
                numericalize(example["sentence"], tokenizer, vocab), dtype=torch.int64
            )
            token_ids_list.append(token_ids)
            offsets.append(token_ids.size(0))

        labels = torch.tensor(label_list, dtype=torch.int64)
        offsets = torch.tensor(offsets[:-1], dtype=torch.int64).cumsum(dim=0)
        token_ids = torch.cat(token_ids_list)
        return labels.to(device), token_ids.to(device), offsets.to(device)

    return collate_batch


def label_distribution(examples):
    counts = Counter(example["label"] for example in examples)
    total = len(examples)
    return {label: {"count": count, "ratio": count / total} for label, count in sorted(counts.items())}


def find_demo_sentence(examples):
    for pattern in (r"[A-Za-z]", r"\d", r"[，。！？：；,.!?]"):
        for example in examples:
            if re.search(pattern, example["sentence"]):
                return example["sentence"]
    return examples[0]["sentence"]


def print_tokenizer_comparison(basic_vocab, improved_vocab):
    rows = [
        ("basic", len(basic_vocab), "single Chinese characters", "No"),
        ("improved", len(improved_vocab), "jieba Chinese words", "Yes"),
    ]
    header = f"{'Tokenizer':<10} {'Vocab Size':>12} {'Chinese Unit':<26} {'Keep EN/Digits/Punct':<20}"
    print("Tokenizer comparison:")
    print(header)
    print("-" * len(header))
    for tokenizer_name, vocab_size, chinese_unit, keep_extra in rows:
        print(f"{tokenizer_name:<10} {vocab_size:>12,} {chinese_unit:<26} {keep_extra:<20}")


train_examples = load_jsonl(TRAIN_PATH)
test_examples = load_jsonl(TEST_PATH)
train_split, valid_split = stratified_split(train_examples, valid_ratio=0.1)

basic_vocab = build_vocab_from_iter(yield_tokens(train_examples, basic_tokenizer), specials=["<unk>"])
improved_vocab = build_vocab_from_iter(yield_tokens(train_examples, improved_tokenizer), specials=["<unk>"])

comparison_sentence = find_demo_sentence(train_examples)
print("Example sentence for tokenizer comparison:")
print(comparison_sentence)
print()
print("Basic tokenizer output:")
print(basic_tokenizer(comparison_sentence))
print()
print("Improved tokenizer output:")
print(improved_tokenizer(comparison_sentence))
print()
print_tokenizer_comparison(basic_vocab, improved_vocab)
print(f"Vocabulary size difference: {len(improved_vocab) - len(basic_vocab):+,}")
print()
print("Dataset sizes:")
print(f"  Train split:      {len(train_split):,}")
print(f"  Validation split: {len(valid_split):,}")
print(f"  Test split:       {len(test_examples):,}")
print()
print("Label distribution:")
print("  Full training data:", label_distribution(train_examples))
print("  Training split:   ", label_distribution(train_split))
print("  Validation split: ", label_distribution(valid_split))
print("  Test split:       ", label_distribution(test_examples))

tokenizer = improved_tokenizer
vocab = improved_vocab
text_pipeline = lambda text: numericalize(text, tokenizer, vocab)

BATCH_SIZE = 64
train_dataset = HumorDataset(train_split)
valid_dataset = HumorDataset(valid_split)
test_dataset = HumorDataset(test_examples)
collate_batch = build_collate_fn(tokenizer, vocab, device)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
    generator=torch.Generator().manual_seed(SEED),
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_batch,
)

sample_labels, sample_token_ids, sample_offsets = next(iter(train_loader))
print()
print("One batch sanity check:")
print(f"  labels shape:    {tuple(sample_labels.shape)}")
print(f"  token_ids shape: {tuple(sample_token_ids.shape)}")
print(f"  offsets shape:   {tuple(sample_offsets.shape)}")
print(f"  First example token ids: {sample_token_ids[sample_offsets[0]:sample_offsets[1]].tolist()}")

### 2. Build the Model

The model uses `nn.EmbeddingBag` for the bag-of-words representation and a fully connected classifier with two hidden layers, which satisfies the assignment requirement.

In [ ]:
class HumorClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dims, num_class, dropout=0.2):
        super().__init__()
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, mode="mean", sparse=False)

        layers = []
        input_dim = embed_dim
        for hidden_dim in hidden_dims:
            layers.append(nn.Linear(input_dim, hidden_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            input_dim = hidden_dim
        layers.append(nn.Linear(input_dim, num_class))
        self.classifier = nn.Sequential(*layers)
        self.init_weights()

    def init_weights(self):
        nn.init.uniform_(self.embedding.weight, -0.5, 0.5)
        for module in self.classifier:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                nn.init.zeros_(module.bias)

    def forward(self, token_ids, offsets):
        embedded = self.embedding(token_ids, offsets)
        return self.classifier(embedded)


EMBED_DIM = 128
HIDDEN_DIMS = [128, 64]
NUM_CLASS = 2
model = HumorClassifier(
    vocab_size=len(vocab),
    embed_dim=EMBED_DIM,
    hidden_dims=HIDDEN_DIMS,
    num_class=NUM_CLASS,
    dropout=0.2,
).to(device)

with torch.no_grad():
    logits = model(sample_token_ids, sample_offsets)

print(model)
print()
print(f"Vocabulary size used for training: {len(vocab):,}")
print(f"Logits shape for one batch: {tuple(logits.shape)}")

### 3. Train and Evaluate

The final model is trained with a fixed random seed, validated on a held-out split from `train.jsonl`, and then evaluated on the provided labeled `test.jsonl` file using accuracy, precision, recall, and F1.

In [ ]:
def compute_metrics(predictions, labels):
    predictions = predictions.cpu()
    labels = labels.cpu()

    tp = int(((predictions == 1) & (labels == 1)).sum().item())
    tn = int(((predictions == 0) & (labels == 0)).sum().item())
    fp = int(((predictions == 1) & (labels == 0)).sum().item())
    fn = int(((predictions == 0) & (labels == 1)).sum().item())

    total = tp + tn + fp + fn
    accuracy = (tp + tn) / total if total else 0.0
    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


def train_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_examples = 0

    for labels, token_ids, offsets in dataloader:
        optimizer.zero_grad()
        logits = model(token_ids, offsets)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_examples += batch_size

    return {
        "loss": total_loss / total_examples,
        "accuracy": total_correct / total_examples,
    }


@torch.no_grad()
def evaluate(model, dataloader, criterion):
    model.eval()
    total_loss = 0.0
    total_examples = 0
    all_predictions = []
    all_labels = []

    for labels, token_ids, offsets in dataloader:
        logits = model(token_ids, offsets)
        loss = criterion(logits, labels)
        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size
        all_predictions.append(predictions)
        all_labels.append(labels)

    predictions = torch.cat(all_predictions)
    labels = torch.cat(all_labels)
    metrics = compute_metrics(predictions, labels)
    metrics["loss"] = total_loss / total_examples
    return metrics


@torch.no_grad()
def predict_text(text, model, text_pipeline):
    model.eval()
    token_ids = torch.tensor(text_pipeline(text), dtype=torch.int64, device=device)
    offsets = torch.tensor([0], dtype=torch.int64, device=device)
    logits = model(token_ids, offsets)
    probabilities = torch.softmax(logits, dim=1).squeeze(0).cpu().tolist()
    prediction = int(torch.argmax(logits, dim=1).item())
    return prediction, probabilities


train_label_counts = Counter(example["label"] for example in train_split)
class_weights = torch.tensor(
    [len(train_split) / (NUM_CLASS * train_label_counts[label]) for label in range(NUM_CLASS)],
    dtype=torch.float32,
    device=device,
)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=0.5, patience=2
)

EPOCHS = 12
best_val_f1 = -1.0
best_state = copy.deepcopy(model.state_dict())
history = []

print(f"Class weights: {class_weights.tolist()}")
print()
for epoch in range(1, EPOCHS + 1):
    train_metrics = train_epoch(model, train_loader, optimizer, criterion)
    valid_metrics = evaluate(model, valid_loader, criterion)
    scheduler.step(valid_metrics["f1"])

    history.append(
        {
            "epoch": epoch,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "valid_loss": valid_metrics["loss"],
            "valid_accuracy": valid_metrics["accuracy"],
            "valid_precision": valid_metrics["precision"],
            "valid_recall": valid_metrics["recall"],
            "valid_f1": valid_metrics["f1"],
            "lr": optimizer.param_groups[0]["lr"],
        }
    )

    if valid_metrics["f1"] > best_val_f1:
        best_val_f1 = valid_metrics["f1"]
        best_state = copy.deepcopy(model.state_dict())

    print(
        f"Epoch {epoch:02d} | "
        f"train loss {train_metrics['loss']:.4f} | "
        f"train acc {train_metrics['accuracy']:.4f} | "
        f"valid loss {valid_metrics['loss']:.4f} | "
        f"valid acc {valid_metrics['accuracy']:.4f} | "
        f"valid precision {valid_metrics['precision']:.4f} | "
        f"valid recall {valid_metrics['recall']:.4f} | "
        f"valid f1 {valid_metrics['f1']:.4f} | "
        f"lr {optimizer.param_groups[0]['lr']:.5f}"
    )

model.load_state_dict(best_state)
final_valid_metrics = evaluate(model, valid_loader, criterion)
final_test_metrics = evaluate(model, test_loader, criterion)

print()
print("Best validation metrics:")
for key in ["loss", "accuracy", "precision", "recall", "f1"]:
    print(f"  {key}: {final_valid_metrics[key]:.4f}")

print()
print("Final test metrics:")
for key in ["accuracy", "precision", "recall", "f1"]:
    print(f"  {key}: {final_test_metrics[key]:.4f}")
print(
    "  confusion counts: "
    f"TP={final_test_metrics['tp']}, TN={final_test_metrics['tn']}, "
    f"FP={final_test_metrics['fp']}, FN={final_test_metrics['fn']}"
)

label_names = {0: "not humor", 1: "humor"}
print()
print("Sample predictions on the test set:")
for example in test_examples[:5]:
    pred_label, probs = predict_text(example["sentence"], model, text_pipeline)
    print(
        f"  {example['id']}: gold={label_names[example['label']]}, "
        f"pred={label_names[pred_label]}, probs={[round(p, 4) for p in probs]}"
    )
    print(f"    text={example['sentence']}")